In [1]:
!pip install -q apache-airflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.5/607.5 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 

In [2]:
import os
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
!airflow db migrate

2026-10-09T15:29:27.001471Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-10-09T15:29:27.746717Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-09T15:29:27.746961Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-09T15:29:28.966002Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-09T15:29:28.966225Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-09T15:29:28.966774Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:749
2026-10-09T15:29:28.966949Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:730
2026-10-09T15:29:28.967113Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow: -4

In [3]:
!mkdir -p /root/airflow/dags

In [4]:
%%writefile /root/airflow/dags/data_extraction.py
# Practical 4: Extracting Data from APIs and Flat Files Using Python
# Objective: fetch user data from a public REST API, clean it, and merge it
#            with a flat CSV file containing location details.
import pandas as pd
import requests

# >>> CHANGE THIS TO YOUR NAME <<<
STUDENT_NAME = "Mohammad Ayaan Sajid Shaikh"


def extract_api_data(url):
    """Fetches data from a REST API and returns a DataFrame."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        # Normalizing JSON data into a flat table
        return pd.json_normalize(data)
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return pd.DataFrame()


def extract_csv_data(file_path):
    """Reads a flat CSV file."""
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError as e:
        print(f"File Error: {e}")
        return pd.DataFrame()


if __name__ == "__main__":
    # 1. API Extraction
    api_url = "https://jsonplaceholder.typicode.com/users"
    api_df = extract_api_data(api_url)
    if not api_df.empty:
        api_df = api_df[["id", "name", "email", "company.name"]]
        api_df.rename(columns={"company.name": "company"}, inplace=True)

    # 2. Flat File Extraction (Mock CSV Data)
    # 'locations.csv' has columns: id, city
    csv_file = "locations.csv"

    # Creating a dummy file for the lab test run
    pd.DataFrame(
        {
            "id": list(range(1, 11)),
            "city": [
                "New York", "London", "Paris", "Tokyo", "Berlin",
                "Delhi", "Sydney", "Moscow", "Cairo", "Beijing",
            ],
        }
    ).to_csv(csv_file, index=False)

    csv_df = extract_csv_data(csv_file)

    # 3. Data Transformation & Merging
    if not api_df.empty and not csv_df.empty:
        merged_df = pd.merge(api_df, csv_df, on="id", how="inner")
        print("\n--- Merged ETL Pipeline Data View ---")
        print(merged_df.head())

        # Save to target Data Lake / Warehouse stage
        merged_df.to_csv("cleaned_warehouse_profiles.csv", index=False)
        print("\nData successfully saved to 'cleaned_warehouse_profiles.csv'")

        # Check for missing values after the merge
        print("\nMissing values per column:\n", merged_df.isnull().sum())

    print("\n" + "=" * 40)
    print("Submitted by:", STUDENT_NAME)
    print("=" * 40)

Writing /root/airflow/dags/data_extraction.py


In [5]:
%%writefile /root/airflow/dags/data_pipeline_dag.py
# Practical 5: Setting up Apache Airflow and Creating a DAG
# Objective: configure a DAG that runs daily with sequential tasks:
#            a start notifier, an extraction script, and a pipeline success logger.
# Place this file in your Airflow DAGs directory: ~/airflow/dags/data_pipeline_dag.py
import os
from datetime import datetime, timedelta

from airflow import DAG

# Works on both Airflow 3.x (new import paths) and Airflow 2.x (old paths)
try:
    from airflow.providers.standard.operators.bash import BashOperator
    from airflow.providers.standard.operators.empty import EmptyOperator
except ImportError:
    from airflow.operators.bash import BashOperator
    from airflow.operators.empty import EmptyOperator

# >>> CHANGE THIS TO YOUR NAME <<<
STUDENT_NAME = "Mohammad Ayaan Sajid Shaikh"

# Practical 4 script: copy it to ~/airflow/dags/ and name it data_extraction.py
EXTRACTION_SCRIPT = os.path.expanduser("~/airflow/dags/data_extraction.py")

# Default arguments for the workflow pipeline execution
default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

# Define the DAG context
with DAG(
    "university_etl_orchestration",
    default_args=default_args,
    description="Lab assignment for API and Flat File ETL orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    # Task 1: Pipeline Init Anchor
    start_pipeline = EmptyOperator(task_id="start_pipeline")

    # Task 2: Run Python Extraction Script
    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command=f"python3 {EXTRACTION_SCRIPT}",
    )

    # Task 3: Finalize and Log Metrics
    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully at $(date)"',
    )

    # Task 4: Print the student name in the task log
    show_student_name = BashOperator(
        task_id="show_student_name",
        bash_command=f'echo "Submitted by: {STUDENT_NAME}"',
    )

    # Task dependencies (execution order)
    start_pipeline >> execute_extraction >> pipeline_complete >> show_student_name

# ---------------------------------------------------------------------------
# Commands to run (Terminal):
#   airflow dags list
#   airflow tasks test university_etl_orchestration start_pipeline 2026-01-01
#   airflow tasks test university_etl_orchestration run_extraction_script 2026-01-01
#   airflow tasks test university_etl_orchestration log_pipeline_success 2026-01-01
#   airflow tasks test university_etl_orchestration show_student_name 2026-01-01
# ---------------------------------------------------------------------------

Writing /root/airflow/dags/data_pipeline_dag.py


In [6]:
!airflow tasks test university_etl_orchestration start_pipeline 2026-01-01

2026-10-09T15:29:41.664606Z [info     ] DAG bundles loaded: dags-folder [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-10-09T15:29:42.430149Z [info     ] Added new DAG bundle dags-folder to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:378
2026-10-09T15:29:42.442788Z [info     ] Filling up the DagBag from /root/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:460
2026-10-09T15:29:42.478260Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:218
2026-10-09T15:29:42.499488Z [info     ] Creating ORM DAG for university_etl_orchestration [airflow.dag_processing.collection] loc=collection.py:102
2026-10-09T15:29:42.515278Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:453 next_info=DagRunInfo(run_after=DateTime(2026, 10, 9, 15, 29, 42, 515139, tzinfo=Timezone('UTC')),

In [7]:
!airflow tasks test university_etl_orchestration run_extraction_script 2026-01-01

2026-10-09T15:29:56.801254Z [info     ] DAG bundles loaded: dags-folder [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-10-09T15:29:57.029167Z [info     ] Filling up the DagBag from /root/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:460
2026-10-09T15:29:57.051073Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:218
2026-10-09T15:29:57.073310Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:453 next_info=DagRunInfo(run_after=DateTime(2026, 10, 9, 15, 29, 57, 73207, tzinfo=Timezone('UTC')), data_interval=DataInterval(start=DateTime(2026, 10, 9, 15, 29, 57, 73207, tzinfo=Timezone('UTC')), end=DateTime(2026, 10, 9, 15, 29, 57, 73207, tzinfo=Timezone('UTC'))), partition_date=None, partition_key=None)
2026-10-09T15:29:57.073673Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.py:8

In [8]:
!airflow tasks test university_etl_orchestration log_pipeline_success 2026-01-01

2026-10-09T15:30:11.713326Z [info     ] DAG bundles loaded: dags-folder [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-10-09T15:30:11.939941Z [info     ] Filling up the DagBag from /root/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:460
2026-10-09T15:30:11.963641Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:218
2026-10-09T15:30:11.986015Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:453 next_info=DagRunInfo(run_after=DateTime(2026, 10, 9, 15, 30, 11, 985912, tzinfo=Timezone('UTC')), data_interval=DataInterval(start=DateTime(2026, 10, 9, 15, 30, 11, 985912, tzinfo=Timezone('UTC')), end=DateTime(2026, 10, 9, 15, 30, 11, 985912, tzinfo=Timezone('UTC'))), partition_date=None, partition_key=None)
2026-10-09T15:30:11.986358Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.p

In [9]:
!airflow tasks test university_etl_orchestration show_student_name 2026-01-01

2026-10-09T15:30:25.776133Z [info     ] DAG bundles loaded: dags-folder [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:304
2026-10-09T15:30:25.999888Z [info     ] Filling up the DagBag from /root/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:460
2026-10-09T15:30:26.025403Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:218
2026-10-09T15:30:26.048859Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:453 next_info=DagRunInfo(run_after=DateTime(2026, 10, 9, 15, 30, 26, 48751, tzinfo=Timezone('UTC')), data_interval=DataInterval(start=DateTime(2026, 10, 9, 15, 30, 26, 48751, tzinfo=Timezone('UTC')), end=DateTime(2026, 10, 9, 15, 30, 26, 48751, tzinfo=Timezone('UTC'))), partition_date=None, partition_key=None)
2026-10-09T15:30:26.049214Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.py:8

In [10]:
!airflow dags list

             |             |             |           |             | bundle_vers
dag_id       | fileloc     | owners      | is_paused | bundle_name | ion        
=============+=============+=============+===========+=============+============
university_e | /root/airfl | data_engine | True      | dags-folder | None       
tl_orchestra | ow/dags/dat | ering_lab   |           |             |            
tion         | a_pipeline_ |             |           |             |            
             | dag.py      |             |           |             |            
                                                                                
